In [0]:
-- ============================================================
-- Szenario A: Sofia Morales (customer_id = 1) wird von Bronze auf
-- Silver hochgestuft -> Nachweis, dass SCD Type 2 funktioniert
-- ============================================================

-- Vorher-Zustand dokumentieren (Screenshot 1)
SELECT customer_id, first_name, last_name, loyalty_tier, effective_from, effective_to, is_current
FROM brewflow_warehouse.dim_customer
WHERE customer_id = 1;
-- Erwartet: genau 1 Zeile, loyalty_tier = Bronze, is_current = true

-- Änderung an der Quelle (OLTP) vornehmen
UPDATE brewflow_oltp.customers
SET loyalty_tier = 'Silver'
WHERE customer_id = 1;

-- ============================================================
-- JETZT: Pipeline erneut ausführen!
-- Entweder die drei gespeicherten Queries einzeln (03 -> 04 -> 05)
-- oder direkt den Lakeflow Job "brewflow_nightly_pipeline" per "Run now"
-- ============================================================

-- Nachher-Zustand prüfen (Screenshot 2)
SELECT customer_id, first_name, last_name, loyalty_tier, effective_from, effective_to, is_current
FROM brewflow_warehouse.dim_customer
WHERE customer_id = 1
ORDER BY effective_from;
-- Erwartet: 2 Zeilen für customer_id 1
--   Zeile 1: Bronze, is_current = false, effective_to = gestern
--   Zeile 2: Silver, is_current = true,  effective_from = heute, effective_to = NULL

-- Gesamtzahl dim_customer sollte jetzt 11 sein (10 Kunden, einer mit 2 Versionen)
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT customer_id) AS distinct_customers
FROM brewflow_warehouse.dim_customer;
-- Erwartet: total_rows = 11, distinct_customers = 10